# Sentence Transformer

In [ ]:
!pip install -q sentence-transformers

## Imports

In [ ]:
import os, re, random, warnings
os.environ["WANDB_MODE"] = "disabled"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A','B','C','D','E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'

## 1. Data Loading & EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

plt.figure(figsize=(6,4))
train_df['answer'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Answer Distribution (Train)')
plt.xlabel('Option')
plt.ylabel('Count')
plt.show()

train_df['opt_len'] = train_df[LABELS].apply(lambda row: row.str.len().mean(), axis=1)
plt.figure(figsize=(6,4))
sns.histplot(train_df['opt_len'], bins=30, kde=True)
plt.title('Average Option Length Distribution')
plt.xlabel('Avg characters')
plt.show()

def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

train_df['option_set'] = train_df.apply(option_set_key, axis=1)
dup_count = train_df.duplicated(subset=['option_set']).sum()
print(f"Number of duplicate option-sets in train: {dup_count}")

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)

## 2. Preprocessing & Strict Splitting

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d:
            uf.union(i, d[k])
        else:
            d[k] = i
train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))
train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l:i for i,l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)
y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero overlap in option-sets)")

## 3. Feature Extraction using Sentence Transformer

In [ ]:
model_name = 'all-MiniLM-L6-v2'
print(f"Loading {model_name}...")
encoder = SentenceTransformer(model_name, device=device)

def get_sentence_embeddings(texts, batch_size=32):
    return encoder.encode(texts, convert_to_tensor=True, show_progress_bar=True, batch_size=batch_size)

def extract_embedding_features(df, batch_size=32):
    prompts = [clean_prompt(row['prompt']) for _, row in df.iterrows()]
    options = []
    for _, row in df.iterrows():
        for l in LABELS:
            options.append(clean_text(row[l]))

    prompt_embs = get_sentence_embeddings(prompts, batch_size)
    opt_embs = get_sentence_embeddings(options, batch_size)

    opt_embs = opt_embs.view(len(df), 5, -1)

    features = []
    for i in range(len(df)):
        p_emb = prompt_embs[i]
        row_feat = []
        for j in range(5):
            o_emb = opt_embs[i, j]
            concat = torch.cat([p_emb, o_emb]).cpu().numpy()
            cos_sim = torch.cosine_similarity(p_emb.unsqueeze(0), o_emb.unsqueeze(0)).item()
            row_feat.extend(list(concat) + [cos_sim])
        features.append(row_feat)
    return np.array(features)

print("Extracting features for train...")
train_feats = extract_embedding_features(train_split)
print("Extracting features for val...")
val_feats = extract_embedding_features(val_split)
print("Extracting features for test...")
test_feats = extract_embedding_features(test_df)

print(f"Feature shape: train {train_feats.shape}, val {val_feats.shape}, test {test_feats.shape}")

## 4. Dimensionality Reduction (PCA)

In [ ]:
# Reduce to 256 components to speed up and reduce memory
pca = PCA(n_components=256, random_state=SEED)
train_feats = pca.fit_transform(train_feats)
val_feats = pca.transform(val_feats)
test_feats = pca.transform(test_feats)
print(f"After PCA: train {train_feats.shape}, val {val_feats.shape}, test {test_feats.shape}")

## 5. Metric: MAP@3

In [ ]:
def map3_from_probs(probs, true_idx):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    true_lab = [LABELS[i] for i in true_idx]
    pred_lab = [" ".join(LABELS[j] for j in row) for row in top3]
    score = 0.0
    for t, p in zip(true_lab, pred_lab):
        for i, c in enumerate(p.split()[:3]):
            if c == t:
                score += 1.0/(i+1); break
    return score / len(true_idx)

## 6. Classifier on Top of Embeddings (5‑Fold Logistic Regression)

In [ ]:
GROUP_NAME = "sentence_transformer_lr"  

gkf = GroupKFold(n_splits=5)
oof_probs = np.zeros((len(train_split), 5))
val_probs_list, test_probs_list = [], []

for fold, (tr_i, va_i) in enumerate(gkf.split(train_feats, y_tr, groups)):
    wandb.init(
        project="smart-mcq-solver",
        entity="23f2004192-dl-genai-project",
        group=GROUP_NAME,
        job_type="cv_fold",
        tags=["sentence-transformer", "logreg"],
        name=f"fold_{fold}",
        config={"model": model_name, "pca_components": 256, "classifier": "LogisticRegression"}
    )

    X_tr, y_tr_f = train_feats[tr_i], y_tr[tr_i]
    X_va, y_va_f = train_feats[va_i], y_tr[va_i]

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)

    clf = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                             max_iter=1000, random_state=SEED, C=1.0)
    clf.fit(X_tr_scaled, y_tr_f)

    fold_tr_probs  = clf.predict_proba(X_tr_scaled)
    fold_val_probs = clf.predict_proba(X_va_scaled)

    fold_map3  = map3_from_probs(fold_val_probs, y_va_f)
    fold_preds = fold_val_probs.argmax(axis=1)
    fold_acc   = accuracy_score(y_va_f, fold_preds)
    fold_f1    = f1_score(y_va_f, fold_preds, average='macro')
    fold_train_loss = log_loss(y_tr_f, fold_tr_probs, labels=list(range(5)))
    fold_val_loss   = log_loss(y_va_f, fold_val_probs, labels=list(range(5)))

    print(f"Fold {fold} val MAP3: {fold_map3:.4f}  Acc: {fold_acc:.4f}  F1: {fold_f1:.4f}")

    wandb.log({
        "val_map3": fold_map3,
        "val_accuracy": fold_acc,
        "val_f1": fold_f1,
        "train_loss": fold_train_loss,
        "val_loss": fold_val_loss,
    })

    oof_probs[va_i] = fold_val_probs
    val_scaled = scaler.transform(val_feats)
    test_scaled = scaler.transform(test_feats)
    val_probs_list.append(clf.predict_proba(val_scaled))
    test_probs_list.append(clf.predict_proba(test_scaled))
    wandb.finish()

val_probs = np.mean(val_probs_list, axis=0)
test_probs = np.mean(test_probs_list, axis=0)
val_true = val_split['label'].values

final_map3 = map3_from_probs(val_probs, val_true)
print(f"Sentence-Transformer + LR Val MAP3: {final_map3:.4f}")

# Final summary run 
final_preds = np.argmax(val_probs, axis=1)
final_acc = accuracy_score(val_true, final_preds)
final_f1  = f1_score(val_true, final_preds, average='macro')

wandb.init(
    project="smart-mcq-solver",
    entity="23f2004192-dl-genai-project",
    group=GROUP_NAME,
    job_type="summary",
    name="final_summary",
    tags=["sentence-transformer", "logreg", "final"],
    config={"model": model_name, "pca_components": 256, "classifier": "LogisticRegression"}
)

wandb.log({
    "final_val_map3": final_map3,
    "final_val_accuracy": final_acc,
    "final_val_f1": final_f1,
})

wandb.finish()

## 7. Evaluation on Validation Set

In [ ]:
val_preds = np.argmax(val_probs, axis=1)
acc = accuracy_score(val_true, val_preds)
f1 = f1_score(val_true, val_preds, average='macro')
print(f"Validation Accuracy: {acc:.4f}, Macro F1: {f1:.4f}")

## 8. Final Submission

In [ ]:
top3 = np.argsort(-test_probs, axis=1)[:, :3]
preds = [(test_df.iloc[i]['id'], " ".join(LABELS[j] for j in top3[i])) for i in range(len(test_df))]
sub = pd.DataFrame(preds, columns=['ID','Prediction']).sort_values('ID').reset_index(drop=True)
sub.to_csv('submission.csv', index=False)
print("\nSubmission saved")
print(sub.head(10))